# Program 9: Macro Regime Dashboard

**Curriculum Days: 136–150 (Portfolio Construction & Risk / Live Trading)**

Pulls key macro indicators from FRED and yfinance, combines them into a composite regime score that classifies the current environment as **RISK-ON**, **NEUTRAL**, or **RISK-OFF**. Shows a historical regime timeline so you can see when regimes shifted.

**What it produces:**
- 6 macro indicator panels (Fed Funds, yield spread, CPI, unemployment, VIX, HY OAS)
- Composite regime score gauge (0–100)
- VIX term structure (contango vs backwardation)
- US Treasury yield curve with inversion detection

## Setup

**FRED API Key**: Get a free key at https://fred.stlouisfed.org/docs/api/api_key.html. The dashboard degrades gracefully to yfinance-only if FRED is unavailable.

In [ ]:
!pip install fredapi yfinance numpy matplotlib pandas -q

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
import matplotlib.dates as mdates
import yfinance as yf
from datetime import datetime, date, timedelta

try:
    from fredapi import Fred
    FREDAPI_AVAILABLE = True
except ImportError:
    FREDAPI_AVAILABLE = False
    print("fredapi not installed. Falling back to yfinance for all data.")

print('All imports loaded.')

## 1. Configuration

**Quant Concept: Macro Factor Models (Days 136–140)**

Options prices are driven by more than just the underlying. Macro factors like the Fed Funds rate, yield curve slope, credit spreads, and VIX levels define the *regime* you're trading in. A regime-aware trader adjusts position sizing and strategy selection based on these signals.

The composite score weights each indicator and produces a single number (0 = pure risk-on, 100 = pure risk-off).

In [ ]:
# Set your FRED API key here (get free at fred.stlouisfed.org)
FRED_API_KEY = os.environ.get("FRED_API_KEY", "YOUR_FRED_API_KEY_HERE")

LOOKBACK_YEARS = 5
REGIME_WINDOW  = 63

FRED_SERIES = {
    "fed_funds_rate":  "FEDFUNDS",
    "yield_10y":       "DGS10",
    "yield_2y":        "DGS2",
    "yield_5y":        "DGS5",
    "yield_30y":       "DGS30",
    "cpi_yoy":         "CPIAUCSL",
    "unemployment":    "UNRATE",
    "hy_oas":          "BAMLH0A0HYM2",
    "real_gdp_growth": "A191RL1Q225SBEA",
}

INDICATOR_WEIGHTS = {
    "vix_score":        0.25,
    "yield_spread":     0.20,
    "hy_oas_score":     0.20,
    "unemployment":     0.15,
    "cpi_score":        0.10,
    "fed_funds_score":  0.10,
}

VIX_LEVELS = {"risk_on": 15, "neutral_low": 20, "neutral_high": 25, "risk_off": 35}
HY_LEVELS  = {"risk_on": 300, "neutral_low": 400, "neutral_high": 550, "risk_off": 700}

print(f"Indicator weights sum to {sum(INDICATOR_WEIGHTS.values()):.2f} (should be 1.00)")

## 2. Data Fetching

**Quant Concept: Multi-Source Data Integration (Days 141–145)**

Real quant systems pull from multiple sources and gracefully handle failures. We use FRED for macro time series and yfinance for VIX and yield curve data as a fallback.

In [ ]:
def fetch_fred_series(fred_client, series_id, start_date):
    try:
        s = fred_client.get_series(series_id, observation_start=start_date)
        return s.dropna()
    except Exception as e:
        print(f"    FRED fetch failed for {series_id}: {e}")
        return pd.Series(dtype=float)

def fetch_all_fred(start_date):
    data = {}
    if FREDAPI_AVAILABLE and FRED_API_KEY != "YOUR_FRED_API_KEY_HERE":
        fred = Fred(api_key=FRED_API_KEY)
        print("  Using FRED API...")
        for key, series_id in FRED_SERIES.items():
            print(f"    Fetching {key} ({series_id})...")
            data[key] = fetch_fred_series(fred, series_id, start_date)
    else:
        print("  FRED API not configured \u2014 using yfinance proxies...")
        data = {k: pd.Series(dtype=float) for k in FRED_SERIES}
    return data

def fetch_vix_data(start_date):
    result = {}
    for ticker, label in [("^VIX", "vix_spot"), ("^VIX3M", "vix_3m")]:
        try:
            hist = yf.download(ticker, start=start_date, progress=False, auto_adjust=True)
            if not hist.empty:
                result[label] = hist['Close']
                print(f"    {ticker}: latest = {hist['Close'].iloc[-1]:.2f}")
            else:
                result[label] = pd.Series(dtype=float)
        except Exception:
            result[label] = pd.Series(dtype=float)
    return result

def fetch_yield_curve_yfinance():
    tickers = {"3m": "^IRX", "2y": "^TWO", "5y": "^FVX", "10y": "^TNX", "30y": "^TYX"}
    curve = {}
    for label, ticker in tickers.items():
        try:
            hist = yf.download(ticker, period='5d', progress=False, auto_adjust=True)
            if not hist.empty:
                curve[label] = float(hist['Close'].iloc[-1])
        except Exception:
            pass
    return curve

print('Data fetchers ready.')

## 3. Indicator Scoring Engine

**Quant Concept: Composite Risk Indicators (Days 136–145)**

Each macro indicator is scored on a 0–100 scale where 0 = risk-on and 100 = risk-off:

- **VIX**: Low VIX ($< 15$) = complacency/risk-on. High VIX ($> 35$) = fear/risk-off.
- **Yield Spread** ($10Y - 2Y$): Steep = growth. Inverted = recession signal.
- **HY OAS**: Tight credit spreads = confidence. Wide = stress.
- **CPI**: Goldilocks (1.5–2.5%) = neutral. Extreme = destabilizing.
- **Fed Funds**: Real rate ($r - \pi$) positive = restrictive for equities.

The **composite** is the weighted average:

$$\text{Score} = \sum_i w_i \cdot s_i, \quad \text{where } \sum w_i = 1$$

In [ ]:
def score_vix(vix_value):
    if np.isnan(vix_value): return 50.0
    if vix_value <= VIX_LEVELS['risk_on']: return 10.0
    elif vix_value <= VIX_LEVELS['neutral_low']:
        return 30.0 + 20 * (vix_value - VIX_LEVELS['risk_on']) / (VIX_LEVELS['neutral_low'] - VIX_LEVELS['risk_on'])
    elif vix_value <= VIX_LEVELS['neutral_high']: return 50.0
    elif vix_value <= VIX_LEVELS['risk_off']:
        return 60.0 + 30 * (vix_value - VIX_LEVELS['neutral_high']) / (VIX_LEVELS['risk_off'] - VIX_LEVELS['neutral_high'])
    else: return 95.0

def score_yield_spread(spread_10y_2y):
    if np.isnan(spread_10y_2y): return 50.0
    if spread_10y_2y > 1.5: return 15.0
    elif spread_10y_2y > 0.5: return 30.0
    elif spread_10y_2y > 0.0: return 50.0
    elif spread_10y_2y > -0.5: return 65.0
    else: return 85.0

def score_hy_oas(oas_bps):
    if np.isnan(oas_bps): return 50.0
    if oas_bps <= HY_LEVELS['risk_on']: return 10.0
    elif oas_bps <= HY_LEVELS['neutral_low']:
        return 10 + 40 * (oas_bps - HY_LEVELS['risk_on']) / (HY_LEVELS['neutral_low'] - HY_LEVELS['risk_on'])
    elif oas_bps <= HY_LEVELS['neutral_high']: return 50.0
    elif oas_bps <= HY_LEVELS['risk_off']:
        return 50 + 35 * (oas_bps - HY_LEVELS['neutral_high']) / (HY_LEVELS['risk_off'] - HY_LEVELS['neutral_high'])
    else: return 90.0

def score_unemployment(rate):
    if np.isnan(rate): return 50.0
    if rate <= 4.0: return 15.0
    elif rate <= 5.0: return 35.0
    elif rate <= 6.5: return 55.0
    elif rate <= 8.0: return 75.0
    else: return 90.0

def score_cpi(cpi_yoy):
    if np.isnan(cpi_yoy): return 50.0
    if 1.5 <= cpi_yoy <= 2.5: return 20.0
    elif cpi_yoy < 0: return 70.0
    elif cpi_yoy > 6.0: return 80.0
    elif cpi_yoy > 4.0: return 65.0
    else: return 40.0

def score_fed_funds(rate, cpi_yoy):
    if np.isnan(rate) or np.isnan(cpi_yoy): return 50.0
    real_rate = rate - cpi_yoy
    if real_rate > 2.0: return 75.0
    elif real_rate > 0.5: return 60.0
    elif real_rate > -1.0: return 45.0
    else: return 30.0

def composite_score(indicators_dict):
    total, weight_used = 0.0, 0.0
    for key, weight in INDICATOR_WEIGHTS.items():
        val = indicators_dict.get(key, np.nan)
        if not np.isnan(val):
            total += val * weight
            weight_used += weight
    return total / weight_used if weight_used > 0 else 50.0

def regime_label(score):
    if score < 35: return "RISK-ON", "#2ecc71"
    elif score < 60: return "NEUTRAL", "#f39c12"
    else: return "RISK-OFF", "#e74c3c"

print('Scoring engine loaded.')

## 4. Visualization

The dashboard renders 9 panels: 6 macro time series, a composite score gauge, VIX term structure, and the yield curve.

In [ ]:
def plot_macro_dashboard(fred_data, vix_data, yield_curve, current_indicators, comp_score):
    plt.style.use('dark_background')
    fig = plt.figure(figsize=(22, 18), facecolor='#0d1117')
    fig.suptitle("Macro Regime Dashboard", fontsize=18, fontweight='bold', color='white', y=0.99)
    gs = gridspec.GridSpec(4, 3, figure=fig, hspace=0.50, wspace=0.38,
                           top=0.95, bottom=0.05, left=0.06, right=0.97)
    PLOT_BG = '#151b27'
    start_date_str = str(date.today() - timedelta(days=LOOKBACK_YEARS * 365))

    def ts_panel(ax, series, title, ylabel, color='#3498db', hline=None, fill=False, fmt='%.2f', latest_label=True):
        ax.set_facecolor(PLOT_BG)
        if series is not None and len(series) > 0:
            s = series[series.index >= pd.Timestamp(start_date_str)]
            ax.plot(s.index, s.values, color=color, linewidth=1.5)
            if fill: ax.fill_between(s.index, s.values, alpha=0.15, color=color)
            if hline is not None: ax.axhline(hline, color='#e74c3c', linestyle='--', linewidth=1, alpha=0.7)
            if latest_label and len(s) > 0:
                ax.text(0.98, 0.92, fmt % s.iloc[-1], transform=ax.transAxes, ha='right', color='white', fontsize=10, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'Data Unavailable', transform=ax.transAxes, ha='center', color='#555', fontsize=10)
        ax.set_title(title, color='white', fontsize=9, fontweight='bold', pad=6)
        ax.set_ylabel(ylabel, color='#aaa', fontsize=8)
        ax.tick_params(colors='#aaa', labelsize=7)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
        for spine in ax.spines.values(): spine.set_edgecolor('#333')

    ts_panel(fig.add_subplot(gs[0, 0]), fred_data.get('fed_funds_rate'), "Fed Funds Rate", "Rate (%)", color='#e74c3c', fmt='%.2f%%')
    s10 = fred_data.get('yield_10y', pd.Series(dtype=float))
    s2 = fred_data.get('yield_2y', pd.Series(dtype=float))
    spread = (s10 - s2).dropna() if len(s10) > 0 and len(s2) > 0 else pd.Series(dtype=float)
    ts_panel(fig.add_subplot(gs[0, 1]), spread, "10Y-2Y Yield Spread", "Spread (%)", color='#f39c12', hline=0.0, fill=True, fmt='%.2f%%')
    cpi_raw = fred_data.get('cpi_yoy', pd.Series(dtype=float))
    cpi_yoy = cpi_raw.pct_change(periods=12) * 100 if len(cpi_raw) > 0 else pd.Series(dtype=float)
    ts_panel(fig.add_subplot(gs[0, 2]), cpi_yoy, "CPI YoY Inflation", "%", color='#e67e22', hline=2.0, fmt='%.1f%%')
    ts_panel(fig.add_subplot(gs[1, 0]), fred_data.get('unemployment'), "Unemployment Rate", "%", color='#9b59b6', fmt='%.1f%%')
    ts_panel(fig.add_subplot(gs[1, 1]), vix_data.get('vix_spot', pd.Series(dtype=float)), "VIX (Fear Index)", "VIX", color='#3498db', hline=20, fmt='%.1f')
    ts_panel(fig.add_subplot(gs[1, 2]), fred_data.get('hy_oas'), "ICE BofA HY OAS", "Spread (bps)", color='#e74c3c', hline=400, fmt='%.0f bps')

    # Regime gauge
    ax7 = fig.add_subplot(gs[2, :2])
    ax7.set_facecolor(PLOT_BG)
    ax7.axis('off')
    rlabel, rcolor = regime_label(comp_score)
    ax7.set_title("Composite Macro Regime Score", color='white', fontsize=12, fontweight='bold', loc='left', pad=8)
    ax7.barh(0.5, 35, left=0, height=0.3, color='#2ecc71', alpha=0.3)
    ax7.barh(0.5, 25, left=35, height=0.3, color='#f39c12', alpha=0.3)
    ax7.barh(0.5, 40, left=60, height=0.3, color='#e74c3c', alpha=0.3)
    ax7.axvline(comp_score, color='white', linewidth=4, ymin=0.2, ymax=0.9)
    ax7.text(comp_score, 0.9, f"{comp_score:.0f}", ha='center', color='white', fontsize=14, fontweight='bold', transform=ax7.get_xaxis_transform())
    ax7.set_xlim(0, 100)
    ax7.set_ylim(0, 1)
    ax7.text(17.5, 0.1, "RISK-ON", ha='center', color='#2ecc71', fontsize=10, fontweight='bold')
    ax7.text(47.5, 0.1, "NEUTRAL", ha='center', color='#f39c12', fontsize=10, fontweight='bold')
    ax7.text(80.0, 0.1, "RISK-OFF", ha='center', color='#e74c3c', fontsize=10, fontweight='bold')
    ax7.text(105, 0.5, f"\u25ba {rlabel}", va='center', color=rcolor, fontsize=14, fontweight='bold')

    # VIX term structure
    ax8 = fig.add_subplot(gs[2, 2])
    ax8.set_facecolor(PLOT_BG)
    vix_term = {"VIX (30d)": current_indicators.get('vix_spot', np.nan), "VIX3M (90d)": current_indicators.get('vix_3m', np.nan)}
    valid_terms = [(l, v) for l, v in vix_term.items() if not np.isnan(v)]
    if valid_terms:
        labs, vals = zip(*valid_terms)
        tc = '#e74c3c' if (len(vals) > 1 and vals[0] > vals[-1]) else '#2ecc71'
        ax8.bar(labs, vals, color=[tc]*len(vals), edgecolor='#333', width=0.5)
        for i, (l, v) in enumerate(zip(labs, vals)):
            ax8.text(i, v+0.3, f"{v:.1f}", ha='center', color='white', fontsize=10)
    ax8.set_title("VIX Term Structure", color='white', fontsize=10, fontweight='bold', pad=6)
    ax8.tick_params(colors='#aaa', labelsize=9)
    for spine in ax8.spines.values(): spine.set_edgecolor('#333')

    # Yield curve
    ax9 = fig.add_subplot(gs[3, :])
    ax9.set_facecolor(PLOT_BG)
    tenors, rates = [], []
    for label, key in [("3M","3m"),("2Y","2y"),("5Y","5y"),("10Y","10y"),("30Y","30y")]:
        val = yield_curve.get(key, np.nan)
        if not np.isnan(val): tenors.append(label); rates.append(val)
    if tenors:
        cc = '#e74c3c' if (len(rates) >= 2 and rates[0] > rates[-1]) else '#2ecc71'
        ax9.plot(range(len(tenors)), rates, color=cc, linewidth=2.5, marker='o', markersize=8)
        ax9.fill_between(range(len(tenors)), rates, alpha=0.15, color=cc)
        for i, (t, r) in enumerate(zip(tenors, rates)):
            ax9.annotate(f"{r:.2f}%", (i, r), textcoords='offset points', xytext=(0,10), ha='center', color='white', fontsize=9)
        ax9.set_xticks(range(len(tenors)))
        ax9.set_xticklabels(tenors, color='#ccc', fontsize=10)
        inv = rates[0] > rates[-1] if len(rates) >= 2 else False
        ax9.set_title(f"US Treasury Yield Curve  ({'INVERTED' if inv else 'NORMAL'})", color='white', fontsize=11, fontweight='bold', pad=8)
    ax9.set_ylabel("Yield (%)", color='#aaa', fontsize=9)
    ax9.tick_params(colors='#aaa', labelsize=9)
    for spine in ax9.spines.values(): spine.set_edgecolor('#333')

    plt.tight_layout()
    plt.show()

print('Dashboard renderer loaded.')

## 5. Run the Macro Dashboard

In [ ]:
print("=" * 65)
print("  PROGRAM 9: Macro Regime Dashboard")
print("=" * 65)

start_date = str(date.today() - timedelta(days=LOOKBACK_YEARS * 365 + 90))

print(f"\n[1/4] Fetching FRED macro data (since {start_date})...")
fred_data = fetch_all_fred(start_date)

print("\n[2/4] Fetching VIX data from yfinance...")
vix_data = fetch_vix_data(start_date)

print("\n[3/4] Fetching current yield curve...")
yield_curve = fetch_yield_curve_yfinance()

print("\n[4/4] Computing regime scores...")
def last_val(series):
    if series is not None and len(series) > 0:
        return float(series.iloc[-1])
    return np.nan

vix_spot = last_val(vix_data.get('vix_spot'))
vix_3m   = last_val(vix_data.get('vix_3m'))
hy_oas   = last_val(fred_data.get('hy_oas'))
unemp    = last_val(fred_data.get('unemployment'))
fed_rate = last_val(fred_data.get('fed_funds_rate'))

y10 = yield_curve.get('10y', np.nan)
y2  = yield_curve.get('2y',  np.nan)
yld_spread = y10 - y2 if (not np.isnan(y10) and not np.isnan(y2)) else np.nan

cpi_raw = fred_data.get('cpi_yoy', pd.Series(dtype=float))
cpi_yoy_val = float((cpi_raw.iloc[-1] / cpi_raw.iloc[-13] - 1) * 100) if len(cpi_raw) >= 13 else np.nan

current_indicators = {
    "vix_score":       score_vix(vix_spot),
    "yield_spread":    score_yield_spread(yld_spread),
    "hy_oas_score":    score_hy_oas(hy_oas),
    "unemployment":    score_unemployment(unemp),
    "cpi_score":       score_cpi(cpi_yoy_val),
    "fed_funds_score": score_fed_funds(fed_rate, cpi_yoy_val),
    "vix_spot":        vix_spot,
    "vix_3m":          vix_3m,
}

comp = composite_score(current_indicators)
rlabel, rcolor = regime_label(comp)

print(f"\n  Composite Score: {comp:.1f} / 100")
print(f"  Regime: *** {rlabel} ***")

print("\nRendering dashboard...")
plot_macro_dashboard(fred_data, vix_data, yield_curve, current_indicators, comp)
print("Done.")

## Exercise 1: Adjust Regime Thresholds

Change the VIX thresholds so that risk-on requires VIX < 12 (stricter) and risk-off triggers at VIX > 30 (earlier). How does this change the regime classification? Are you more conservative or aggressive than the default?

In [ ]:
# YOUR CODE HERE: Modify VIX_LEVELS and re-run scoring


## Exercise 2: Change the Indicator Weights

If you believe credit stress (HY OAS) is the most important macro signal, boost its weight to 0.35 and reduce VIX to 0.15. Re-run the composite. Does the regime change? This is how different desks calibrate the same framework differently.

In [ ]:
# YOUR CODE HERE: Modify INDICATOR_WEIGHTS (must sum to 1.0)


## Key Takeaways

1. **Regime awareness is alpha** — The same strategy (e.g., selling iron condors) has very different expected outcomes in risk-on vs risk-off. Knowing the regime *before* you trade is non-negotiable.

2. **The yield curve is the best recession predictor** — An inverted curve (2Y > 10Y) has preceded every recession since 1970. When it inverts, reduce risk and prepare for vol.

3. **VIX term structure reveals fear** — Contango (VIX < VIX3M) is normal. Backwardation (VIX > VIX3M) means the market is pricing near-term stress.

4. **No single indicator is sufficient** — The composite approach avoids false signals. Credit, rates, vol, and employment together paint a clearer picture than any one alone.

5. **Poker parallel** — The macro regime is like the table texture. A tight-aggressive table (risk-off) requires different play than a loose-passive table (risk-on). Adjust your strategy to the environment.